# Indian Market Surveillance Analysis
### Edible Oil Quality Assessment — Assignment 1

## SETUP 

In [13]:
import pandas as pd
import re
import os
from difflib import get_close_matches

# Load all three sheets 
DATA_FILE = "Assignment_1_DATA_SETS.xlsx"

df_c1   = pd.read_excel(DATA_FILE, sheet_name="Sheet_C1 FORM")
df_t1   = pd.read_excel(DATA_FILE, sheet_name="Sheet_T1 FORM ")
df_eval = pd.read_excel(DATA_FILE, sheet_name="Evaluation Criteria_Parameters ")

print(f"C1  rows: {len(df_c1)},  columns: {df_c1.shape[1]}")
print(f"T1  rows: {len(df_t1)},  columns: {df_t1.shape[1]}")
print(f"Eval rows: {len(df_eval)}")
print("\nAll sheets loaded successfully.")

C1  rows: 400,  columns: 19
T1  rows: 400,  columns: 28
Eval rows: 19

All sheets loaded successfully.


---
## Q1 — Data Cleaning & Standardization
**What am I doing :-** Fixing inconsistent oil names in C1 — casing, extra spaces, merged words, spelling variants.

**Logic:**
1. Strip leading/trailing whitespace
2. Collapse internal multiple spaces
3. Lowercase for comparison
4. Remove all spaces to catch `soyabeanoil` → `soybeanoil` style merges
5. Fuzzy-match against a canonical list using `difflib.get_close_matches` (80% similarity cutoff)
6. If fuzzy match fails, apply a manual override map for known edge cases

**Output file:** `Q1_Standardized_Oil_Types.xlsx`

In [15]:
# Step 1: Define canonical (correct) oil names 
# These are the authoritative names. Add/edit here if new oil types appear.
CANONICAL_OILS = [
    "Mustard Oil",
    "Sunflower Seed Oil",
    "Groundnut Oil",
    "Soybean Oil",
    "Rice Bran Oil",
    "Olive Oil",
    "Coconut Oil",
    "Multi-Source Edible Oil",
    "Palm Oil",
    "Cotton Seed Oil",
    "Sesame Oil",
    "Refined Vegetable Oil",
]

# Step 2: Manual overrides for known tricky cases 
# Key = lowercase no-space version of raw value → Value = canonical name
MANUAL_OVERRIDES = {
    "refinedvegetbleoil" : "Refined Vegetable Oil",
    "refinedvegtoil"     : "Refined Vegetable Oil",
    "refinedvegetoil"    : "Refined Vegetable Oil",
    "palmmoil"           : "Palm Oil",
    "groundnutsoil"      : "Groundnut Oil",
    "groundnutsoils"     : "Groundnut Oil",
}

# Precompute lowercase / no-space versions of canonical names for matching
_canonical_lower   = [c.lower() for c in CANONICAL_OILS]
_canonical_nospace = [re.sub(r"\s+", "", c.lower()) for c in CANONICAL_OILS]


def standardize_oil(raw):
    """Return the canonical oil name for a raw string."""
    if pd.isna(raw):
        return raw

    # Collapse spaces, strip edges
    cleaned = re.sub(r"\s+", " ", str(raw)).strip()
    lower   = cleaned.lower()
    nospace = re.sub(r"\s+", "", lower)

    # Manual override check first
    if nospace in MANUAL_OVERRIDES:
        return MANUAL_OVERRIDES[nospace]

    # Fuzzy match on no-space version (catches 'soyabeanoil')
    m = get_close_matches(nospace, _canonical_nospace, n=1, cutoff=0.80)
    if m:
        return CANONICAL_OILS[_canonical_nospace.index(m[0])]

    # Fuzzy match on spaced version
    m = get_close_matches(lower, _canonical_lower, n=1, cutoff=0.80)
    if m:
        return CANONICAL_OILS[_canonical_lower.index(m[0])]

    # Fallback: title-case the cleaned string
    return cleaned.title()


# Step 3: Apply standardization 
df_c1["Oil_Type_Standardized"] = df_c1["Q17. Type of Oil"].apply(standardize_oil)

# Step 4: Build before/after comparison 
comparison = (
    df_c1[["Q17. Type of Oil", "Oil_Type_Standardized"]]
    .drop_duplicates()
    .assign(Changed=lambda x: x["Q17. Type of Oil"] != x["Oil_Type_Standardized"])
    .sort_values("Changed", ascending=False)
    .reset_index(drop=True)
)

print("=== Before vs After Standardization ===")
print(comparison.to_string())
print(f"\nTotal distinct raw values:   {len(comparison)}")
print(f"Values that were changed:     {comparison['Changed'].sum()}")

# Step 5: Export 
out_q1 = "Q1_Standardized_Oil_Types.xlsx"
with pd.ExcelWriter(out_q1, engine="openpyxl") as writer:
    comparison.to_excel(writer, sheet_name="Before_After_Mapping", index=False)
    df_c1[["Q41. Sample Unique Code", "Q17. Type of Oil", "Oil_Type_Standardized"]].to_excel(
        writer, sheet_name="Full_C1_With_Standardized", index=False
    )

print(f"\nSaved → {out_q1}")

=== Before vs After Standardization ===
           Q17. Type of Oil    Oil_Type_Standardized  Changed
0       ground   nut    oil            Groundnut Oil     True
1              soyabean OIL              Soybean Oil     True
2               coconut oil              Coconut Oil     True
3                   PALMOIL                 Palm Oil     True
4                coconutoil              Coconut Oil     True
5                SESAME OIL               Sesame Oil     True
6                 PALMM OIL                 Palm Oil     True
7       Refinedvegetableoil    Refined Vegetable Oil     True
8       REFINEDVEGETBLE OIL    Refined Vegetable Oil     True
9                Sesame OIL               Sesame Oil     True
10              SOYABEANOIL              Soybean Oil     True
11                PALM  OIL                 Palm Oil     True
12              Coconut oil              Coconut Oil     True
13               coconutoil              Coconut Oil     True
14    Refined vegetable Oil   

---
## Q2 — Descriptive Analysis
**Task:**
- Oil-wise sample counts (all oils)
- City-wise distribution for top 5 oil types

**Output file:** `Q2_Descriptive_Analysis.xlsx`

In [16]:
# Oil-wise sample counts 
oil_counts = (
    df_c1["Oil_Type_Standardized"]
    .value_counts()
    .reset_index()
)
oil_counts.columns = ["Oil_Type", "Sample_Count"]
oil_counts["Percentage"] = (oil_counts["Sample_Count"] / len(df_c1) * 100).round(2)

print("=== Oil-wise Sample Counts ===")
print(oil_counts.to_string(index=False))

# City-wise distribution for top 5 oils 
TOP_N = 5
top_oils = oil_counts["Oil_Type"].head(TOP_N).tolist()
print(f"\nTop {TOP_N} oils: {top_oils}")

city_dist = (
    df_c1[df_c1["Oil_Type_Standardized"].isin(top_oils)]
    .groupby(["Oil_Type_Standardized", "Q9. City/District"])
    .size()
    .reset_index(name="Count")
    .sort_values(["Oil_Type_Standardized", "Count"], ascending=[True, False])
    .reset_index(drop=True)
)
city_dist.columns = ["Oil_Type", "City", "Count"]

print("\n=== City-wise Distribution (Top 5 Oils) ===")
print(city_dist.to_string(index=False))

# Export 
out_q2 = "Q2_Descriptive_Analysis.xlsx"
with pd.ExcelWriter(out_q2, engine="openpyxl") as writer:
    oil_counts.to_excel(writer, sheet_name="Oil_Sample_Counts", index=False)
    city_dist.to_excel(writer, sheet_name="City_Distribution_Top5", index=False)

print(f"\nSaved → {out_q2}")

=== Oil-wise Sample Counts ===
               Oil_Type  Sample_Count  Percentage
            Mustard Oil            53       13.25
            Soybean Oil            49       12.25
          Groundnut Oil            42       10.50
     Sunflower Seed Oil            39        9.75
          Rice Bran Oil            34        8.50
            Coconut Oil            33        8.25
             Sesame Oil            30        7.50
               Palm Oil            28        7.00
              Olive Oil            26        6.50
Multi-Source Edible Oil            25        6.25
  Refined Vegetable Oil            23        5.75
        Cotton Seed Oil            18        4.50

Top 5 oils: ['Mustard Oil', 'Soybean Oil', 'Groundnut Oil', 'Sunflower Seed Oil', 'Rice Bran Oil']

=== City-wise Distribution (Top 5 Oils) ===
          Oil_Type                    City  Count
     Groundnut Oil               Faridabad      3
     Groundnut Oil                    Dhar      2
     Groundnut Oil      

---
## Q3 — Pass / Fail Determination
**Task:** Rule-based Pass/Fail system using the Evaluation Criteria sheet.

**Logic per parameter:**
- **Heavy metals / Aflatoxins (numeric max):** value > threshold → FAIL. BLQ/ND/LOQ variants → PASS.
- **Qualitative tests (Polybromide, Argemone, Mineral oil):** `Positive` → FAIL, `Negative` → PASS.
- **Saponification / Iodine / Unsaponifiable / Acid value:** numeric > MAX → FAIL.
- **Configurable:** all thresholds are in the `THRESHOLDS` dict — change one number and it applies everywhere.

**Output file:** `Q3_PassFail_Results.xlsx`

In [17]:
# Threshold config — change values here when regulations change 
# To update a rule: edit the number in "max" or the strings in "qualitative_fail"
THRESHOLDS = {
    # Heavy metals — max 5 PPM; BLQ/ND/LOQ variants always PASS
    "Q10. Lead [Parts Per Million (mg/kg or  mg/l)]"       : {"max": 5},
    "Q11. Copper [ Parts Per Million (mg/kg or  mg/l) ]"   : {"max": 5},
    "Q12. Arsenic [ Parts Per Million (mg/kg or  mg/l) ]"  : {"max": 5},
    "Q13. Tin [ Parts Per Million (mg/kg or  mg/l) ]"      : {"max": 5},
    "Q14. Cadmium [ Parts Per Million (mg/kg or  mg/l) ]"  : {"max": 5},
    "Q15. Mercury [ Parts Per Million (mg/kg or  mg/l) ]"  : {"max": 5},
    # Aflatoxins
    "Q16. Aflatoxin B1 [ µg/kg ]"                          : {"max": 1},
    "Q17. Total Aflatoxins [ µg/kg ]"                      : {"max": 5},
    # Physical/chemical — numeric max
    "Q20. Saponification value [mg KOH/g oil]"             : {"max": 180},
    "Q22. Iodine value"                                    : {"max": 120},
    "Q23. Unsaponifiable matter [% by weight]"             : {"max": 130},
    "Q25. Acid value"                                      : {"max": 3},
    # Qualitative tests — Positive = FAIL
    "Q21. Polybromide test"                                : {"qualitative_fail": ["positive"]},
    "Q28. Test for Argemone oil"                           : {"qualitative_fail": ["positive"]},
    "Q29. Test for Mineral oil"                            : {"qualitative_fail": ["positive"]},
}

# Strings that always mean PASS for numeric columns (Below Limit of Quantification etc.)
QUALITATIVE_PASS_TOKENS = ["blq", "nd", "loq", "<loq", "not detected", "<"]


def is_qualitative_pass(val_str):
    """Return True if the string value means 'not detected / below limit'."""
    v = val_str.strip().lower()
    return any(token in v for token in QUALITATIVE_PASS_TOKENS)


def evaluate_sample(row):
    """Return (Pass_Fail, Failure_Reason) for one T1 row."""
    failures = []

    for col, rules in THRESHOLDS.items():
        if col not in row.index:
            continue
        val = row[col]

        # Missing data → skip (cannot determine pass/fail)
        if pd.isna(val):
            continue

        val_str = str(val).strip()

        # Qualitative test (Positive/Negative) 
        if "qualitative_fail" in rules:
            if val_str.lower() in rules["qualitative_fail"]:
                failures.append(f"{col} = {val_str} (expected Negative)")
            continue

        # Numeric max check 
        if "max" in rules:
            # BLQ/ND/LOQ strings → always pass
            if is_qualitative_pass(val_str):
                continue
            # Try to extract numeric value
            try:
                num = float(val)
                if num > rules["max"]:
                    failures.append(f"{col} = {num:.4f} (max allowed: {rules['max']})")
            except (ValueError, TypeError):
                # Non-numeric and not a known pass token — flag as unresolved
                failures.append(f"{col} = '{val_str}' (unreadable value)")

    status = "FAIL" if failures else "PASS"
    reason = " | ".join(failures)
    return pd.Series({"Pass_Fail": status, "Failure_Reason": reason})


# Apply to all T1 rows 
results = df_t1.apply(evaluate_sample, axis=1)
df_t1 = pd.concat([df_t1, results], axis=1)

# Summary 
print("=== Pass/Fail Summary ===")
print(df_t1["Pass_Fail"].value_counts().to_string())
print(f"\nTotal samples: {len(df_t1)}")
print(f"Pass rate: {(df_t1['Pass_Fail']=='PASS').mean()*100:.1f}%")

print("\n=== Sample FAIL records ===")
fail_df = df_t1[df_t1["Pass_Fail"] == "FAIL"][["Sample Unique Code", "Q6. Type of Oil [By Sampler]", "Pass_Fail", "Failure_Reason"]]
print(fail_df.head(10).to_string(index=False))

# Export 
out_q3 = "Q3_PassFail_Results.xlsx"
with pd.ExcelWriter(out_q3, engine="openpyxl") as writer:
    df_t1.to_excel(writer, sheet_name="T1_With_PassFail", index=False)
    fail_df.to_excel(writer, sheet_name="FAIL_Records_Only", index=False)
    # Summary counts
    summary = df_t1["Pass_Fail"].value_counts().reset_index()
    summary.columns = ["Result", "Count"]
    summary["Percentage"] = (summary["Count"] / len(df_t1) * 100).round(2)
    summary.to_excel(writer, sheet_name="Summary", index=False)

print(f"\nSaved → {out_q3}")

=== Pass/Fail Summary ===
Pass_Fail
FAIL    389
PASS     11

Total samples: 400
Pass rate: 2.8%

=== Sample FAIL records ===
 Sample Unique Code Q6. Type of Oil [By Sampler] Pass_Fail                                                                                                                                                                                                                                                                                                                                                                                                                        Failure_Reason
               1519                    Olive Oil      FAIL                                                                                                                                                                                                                                                                          Q11. Copper [ Parts Per Million (mg/kg or  mg/l) ] = 9.5607 (max allow

---
## Q4 — Sampler vs Lab Mismatch Detection
**Task:** Compare oil type reported by sampler vs identified by lab. Classify as Match / Minor / Major mismatch.

**Classification logic:**
- **Match:** Both refer to the same canonical oil
- **Minor mismatch:** Same oil family (e.g. `Sunflower Oil` vs `Sunflower Seed Oil`, or `Refined Vegetable Oil` vs `Multi-Source Edible Oil`)
- **Major mismatch:** Completely different oils (e.g. `Olive Oil` vs `Mustard Oil`)

**Output file:** `Q4_Mismatch_Report.xlsx`

In [18]:
# Standardize sampler and lab oil names in T1 
# Reuse the same standardize_oil() function from Q1
df_t1["Sampler_Oil_Std"] = df_t1["Q6. Type of Oil [By Sampler]"].apply(standardize_oil)
df_t1["Lab_Oil_Std"]     = df_t1["Q8. Type of Oil by Lab"].apply(standardize_oil)

# Define oil families for minor/major classification 
# Oils in the same family = Minor mismatch (related products)
# Oils in different families = Major mismatch
OIL_FAMILIES = {
    "sunflower"   : ["Sunflower Seed Oil", "Sunflower Oil"],
    "soy"         : ["Soybean Oil"],
    "mustard"     : ["Mustard Oil"],
    "groundnut"   : ["Groundnut Oil"],
    "rice"        : ["Rice Bran Oil"],
    "palm"        : ["Palm Oil"],
    "coconut"     : ["Coconut Oil"],
    "sesame"      : ["Sesame Oil"],
    "cotton"      : ["Cotton Seed Oil"],
    "olive"       : ["Olive Oil"],
    "blended"     : ["Refined Vegetable Oil", "Multi-Source Edible Oil"],
}

def get_oil_family(oil_name):
    for family, members in OIL_FAMILIES.items():
        if oil_name in members:
            return family
    return str(oil_name).lower().strip()   # Unknown → treated as its own family


def classify_mismatch(row):
    s = row["Sampler_Oil_Std"]
    l = row["Lab_Oil_Std"]
    if pd.isna(s) or pd.isna(l):
        return "Cannot Determine"
    if s == l:
        return "Match"
    if get_oil_family(s) == get_oil_family(l):
        return "Minor Mismatch"   # Same family, different name variant
    return "Major Mismatch"       # Completely different oils


df_t1["Mismatch_Type"] = df_t1.apply(classify_mismatch, axis=1)

# Overall summary 
print("=== Overall Mismatch Summary ===")
print(df_t1["Mismatch_Type"].value_counts().to_string())

# Mismatch records only 
mismatch_df = df_t1[df_t1["Mismatch_Type"].isin(["Minor Mismatch", "Major Mismatch"])].copy()

print(f"\nTotal mismatches: {len(mismatch_df)}")
print(mismatch_df[["Sample Unique Code", "Sampler_Oil_Std", "Lab_Oil_Std", "Mismatch_Type"]].head(10).to_string(index=False))

# By oil type 
by_oil = (
    mismatch_df.groupby(["Sampler_Oil_Std", "Mismatch_Type"])
    .size()
    .reset_index(name="Count")
    .sort_values("Count", ascending=False)
)
print("\n Mismatches by Oil Type ")
print(by_oil.to_string(index=False))

# By city 
city_col = "City/District from which the sample was\xa0picked"
by_city = (
    mismatch_df.groupby([city_col, "Mismatch_Type"])
    .size()
    .reset_index(name="Count")
    .sort_values("Count", ascending=False)
)
print("\n Mismatches by City ")
print(by_city.to_string(index=False))

# Export 
out_q4 = "Q4_Mismatch_Report.xlsx"
with pd.ExcelWriter(out_q4, engine="openpyxl") as writer:
    df_t1[["Sample Unique Code", "Q6. Type of Oil [By Sampler]", "Sampler_Oil_Std",
           "Q8. Type of Oil by Lab", "Lab_Oil_Std", "Mismatch_Type"]].to_excel(
        writer, sheet_name="All_Samples_Classified", index=False)
    mismatch_df[["Sample Unique Code", "Sampler_Oil_Std", "Lab_Oil_Std",
                 "Mismatch_Type", city_col]].to_excel(
        writer, sheet_name="Mismatch_Records", index=False)
    by_oil.to_excel(writer, sheet_name="By_Oil_Type", index=False)
    by_city.to_excel(writer, sheet_name="By_City", index=False)

print(f"\nSaved → {out_q4}")

=== Overall Mismatch Summary ===
Mismatch_Type
Major Mismatch    358
Match              42

Total mismatches: 358
 Sample Unique Code    Sampler_Oil_Std             Lab_Oil_Std  Mismatch_Type
               1519          Olive Oil   Refined Vegetable Oil Major Mismatch
               3189          Olive Oil Multi-Source Edible Oil Major Mismatch
               3477          Olive Oil             Mustard Oil Major Mismatch
               5711          Olive Oil      Sunflower Seed Oil Major Mismatch
               5092          Olive Oil           Rice Bran Oil Major Mismatch
               4366        Soybean Oil           Groundnut Oil Major Mismatch
               1098        Soybean Oil           Groundnut Oil Major Mismatch
               5137 Sunflower Seed Oil                Palm Oil Major Mismatch
               4713 Sunflower Seed Oil             Soybean Oil Major Mismatch
               5023        Coconut Oil   Refined Vegetable Oil Major Mismatch

 Mismatches by Oil Type 
  

---
## Q5 — C1 ↔ T1 Interconnection & Key Insights
**Tasks:**
1. Identify correct join keys
2. Build master dataset
3. Analyze delays and operational bottlenecks
4. Derive insights only possible via C1–T1 linkage

**Join key:** `Q41. Sample Unique Code` (C1) ↔ `Sample Unique Code` (T1)

**Output file:** `Q5_Master_Dataset.xlsx`

In [19]:
# Step 1: Verify join keys before merging 
c1_keys = set(df_c1["Q41. Sample Unique Code"].dropna().astype(int))
t1_keys = set(df_t1["Sample Unique Code"].dropna().astype(int))

print(f"C1 unique keys : {len(c1_keys)}")
print(f"T1 unique keys : {len(t1_keys)}")
print(f"Overlapping    : {len(c1_keys & t1_keys)}")
print(f"Only in C1     : {len(c1_keys - t1_keys)}")
print(f"Only in T1     : {len(t1_keys - c1_keys)}")

C1 unique keys : 377
T1 unique keys : 374
Overlapping    : 51
Only in C1     : 326
Only in T1     : 323


In [20]:
# Step 2: Build master dataset 
# Rename C1 join key to match T1
df_c1_join = df_c1.rename(columns={"Q41. Sample Unique Code": "Sample Unique Code"})

master = pd.merge(
    df_c1_join,
    df_t1,
    on="Sample Unique Code",
    how="inner",
    suffixes=("_C1", "_T1")
)

print(f"Master dataset shape: {master.shape}")
print(f"Rows matched: {len(master)} samples have both C1 and T1 records")
print(master[["Sample Unique Code", "Q8. State", "Q9. City/District", "Pass_Fail"]].head(5).to_string(index=False))

Master dataset shape: (53, 52)
Rows matched: 53 samples have both C1 and T1 records
 Sample Unique Code        Q8. State       Q9. City/District Pass_Fail
               5329 Himachal Pradesh                   Mandi      FAIL
               2792     Chhattisgarh                  Raipur      FAIL
               2736     Chhattisgarh Bilaspur - Chhattisgarh      FAIL
               1869   Madhya Pradesh                   Betul      FAIL
               1869   Madhya Pradesh                   Betul      FAIL


In [21]:
# Step 3: Shelf life analysis (delay / operational bottleneck proxy) 
# Manufacture → Expiry gap tells us shelf life; short gaps on already-collected
# samples may indicate delayed testing (operational bottleneck).

master["Mfg_Date"]    = pd.to_datetime(master["Q23. Date of Manufacturing"], dayfirst=True, errors="coerce")
master["Expiry_Date"] = pd.to_datetime(master["Q25. Date of Expiry"],        dayfirst=True, errors="coerce")

master["Shelf_Life_Days"] = (master["Expiry_Date"] - master["Mfg_Date"]).dt.days

# Remaining shelf life at time of collection (proxy for how fresh samples are)
# Using 2025-01-01 as a reference collection period mid-point
REF_DATE = pd.Timestamp("2025-01-01")
master["Remaining_Shelf_Life_Days"] = (master["Expiry_Date"] - REF_DATE).dt.days

# Flag samples with less than 90 days remaining — potential testing delay risk
master["Short_Shelf_Life_Flag"] = master["Remaining_Shelf_Life_Days"] < 90

print("Shelf Life Statistics (days) ")
print(master["Shelf_Life_Days"].describe().round(1).to_string())

print("\nShort Shelf Life Flags (< 90 days remaining at collection) ")
print(master["Short_Shelf_Life_Flag"].value_counts().to_string())

print("\nAverage Shelf Life by Oil Type")
shelf_by_oil = (
    master.groupby("Oil_Type_Standardized")["Shelf_Life_Days"]
    .agg(["mean", "min", "max", "count"])
    .round(1)
    .sort_values("mean", ascending=False)
)
print(shelf_by_oil.to_string())

Shelf Life Statistics (days) 
count      53.0
mean      334.5
std       202.8
min       170.0
25%       270.0
50%       272.0
75%       364.0
max      1096.0

Short Shelf Life Flags (< 90 days remaining at collection) 
Short_Shelf_Life_Flag
False    53

Average Shelf Life by Oil Type
                           mean   min   max  count
Oil_Type_Standardized                             
Olive Oil                1095.3  1095  1096      3
Coconut Oil               390.4   180   560      5
Multi-Source Edible Oil   364.0   364   364      1
Sesame Oil                354.3   334   365      3
Rice Bran Oil             327.6   272   364      5
Mustard Oil               298.3   242   366      6
Groundnut Oil             272.2   181   364      4
Cotton Seed Oil           272.0   272   272      2
Sunflower Seed Oil        264.0   243   272      5
Refined Vegetable Oil     257.2   242   273      4
Soybean Oil               253.0   170   273     13
Palm Oil                  181.0   181   181      2


In [22]:
# Step 4: Key Insights only possible via C1+T1 linkage 

city_col_t1 = "City/District from which the sample was\xa0picked"

#  OPERATIONAL INSIGHT: Which cities have the highest FAIL rates? 
# City comes from C1; Pass_Fail comes from T1 — neither alone can answer this.
city_pf = (
    master.groupby("Q9. City/District")["Pass_Fail"]
    .value_counts(normalize=True)
    .mul(100).round(1)
    .reset_index(name="Percentage")
)
city_fail_rate = (
    city_pf[city_pf["Pass_Fail"] == "FAIL"]
    .sort_values("Percentage", ascending=False)
    .reset_index(drop=True)
)
print("=== OPERATIONAL INSIGHT: City-wise FAIL Rate (%) ===")
print(city_fail_rate.to_string(index=False))

# ANALYTICAL INSIGHT: Which shop types sell the most failing products? 
# Shop type comes from C1 (Q12); Pass_Fail from T1 — linked via join.
shop_pf = (
    master.groupby("Q12. Type of Shop")["Pass_Fail"]
    .value_counts(normalize=True)
    .mul(100).round(1)
    .reset_index(name="Percentage")
)
shop_fail_rate = (
    shop_pf[shop_pf["Pass_Fail"] == "FAIL"]
    .sort_values("Percentage", ascending=False)
    .reset_index(drop=True)
)
print("\n=== ANALYTICAL INSIGHT: Fail Rate by Shop Type (%) ===")
print(shop_fail_rate.to_string(index=False))

# ANALYTICAL INSIGHT: Mismatch + Fail overlap 
# Samples where lab identified a DIFFERENT oil AND that sample also FAILED
double_problem = master[
    (master["Mismatch_Type"] == "Major Mismatch") & (master["Pass_Fail"] == "FAIL")
]
print(f"\n ANALYTICAL INSIGHT: Major Mismatch + FAIL overlap ")
print(f"Samples with both Major Mismatch AND FAIL: {len(double_problem)}")
if len(double_problem) > 0:
    print(double_problem[["Sample Unique Code", "Sampler_Oil_Std", "Lab_Oil_Std", "Pass_Fail", "Failure_Reason"]].to_string(index=False))

=== OPERATIONAL INSIGHT: City-wise FAIL Rate (%) ===
      Q9. City/District Pass_Fail  Percentage
             Ahmednagar      FAIL       100.0
               Amravati      FAIL       100.0
              Baleshwar      FAIL       100.0
           Banas Kantha      FAIL       100.0
                  Betul      FAIL       100.0
               Bhilwara      FAIL       100.0
                 Bhopal      FAIL       100.0
                    Bid      FAIL       100.0
                Bikaner      FAIL       100.0
Bilaspur - Chhattisgarh      FAIL       100.0
               Dehradun      FAIL       100.0
                   Dhar      FAIL       100.0
              Ghaziabad      FAIL       100.0
               Haridwar      FAIL       100.0
              Jaisalmer      FAIL       100.0
                Jalgaon      FAIL       100.0
                 Karnal      FAIL       100.0
               Kolhapur      FAIL       100.0
                    Leh      FAIL       100.0
                  Mandi    

In [23]:
# Export master dataset and all insights 
out_q5 = "Q5_Master_Dataset.xlsx"
with pd.ExcelWriter(out_q5, engine="openpyxl") as writer:
    master.to_excel(writer, sheet_name="Master_Dataset", index=False)

    city_fail_rate.to_excel(writer, sheet_name="Insight_City_Fail_Rate", index=False)
    shop_fail_rate.to_excel(writer, sheet_name="Insight_Shop_Fail_Rate", index=False)
    shelf_by_oil.reset_index().to_excel(writer, sheet_name="Shelf_Life_By_Oil", index=False)

    if len(double_problem) > 0:
        double_problem[["Sample Unique Code", "Sampler_Oil_Std", "Lab_Oil_Std",
                        "Pass_Fail", "Failure_Reason"]].to_excel(
            writer, sheet_name="Mismatch_And_Fail", index=False)

print(f"Saved → {out_q5}")

Saved → Q5_Master_Dataset.xlsx


---
## Final Summary — All Output Files

In [24]:
output_files = [
    ("Q1_Standardized_Oil_Types.xlsx", "Q1", "Oil name standardization mapping"),
    ("Q2_Descriptive_Analysis.xlsx",   "Q2", "Oil counts + city distribution"),
    ("Q3_PassFail_Results.xlsx",        "Q3", "T1 with Pass/Fail + Failure Reason"),
    ("Q4_Mismatch_Report.xlsx",         "Q4", "Sampler vs Lab mismatch report"),
    ("Q5_Master_Dataset.xlsx",          "Q5", "Joined C1+T1 master + insights"),
]

print(f"{'File':<42} {'Q':>2}  {'Status':<10}")
for fname, q, desc in output_files:
    exists = os.path.isfile(fname)
    status = "✓ Created" if exists else "✗ Missing"
    size   = f"{os.path.getsize(fname)//1024} KB" if exists else ""
    print(f"{fname:<42} {q:>2}  {status}  {size}")
    print(f"  → {desc}")

File                                        Q  Status    
Q1_Standardized_Oil_Types.xlsx             Q1  ✓ Created  13 KB
  → Oil name standardization mapping
Q2_Descriptive_Analysis.xlsx               Q2  ✓ Created  9 KB
  → Oil counts + city distribution
Q3_PassFail_Results.xlsx                   Q3  ✓ Created  100 KB
  → T1 with Pass/Fail + Failure Reason
Q4_Mismatch_Report.xlsx                    Q4  ✓ Created  26 KB
  → Sampler vs Lab mismatch report
Q5_Master_Dataset.xlsx                     Q5  ✓ Created  31 KB
  → Joined C1+T1 master + insights
